# Miami Open – Odds Data EDA

**Goal:** Understand the raw JSON structure returned by The-Odds-API for `tennis_atp_miami_open` before designing the ETL pipeline.

**Source:** `data/raw/tennis_odds_<timestamp>.json` (extracted locally via `src/ingestion/extract_odds.py`)

**Questions to answer:**
1. What does the top-level structure look like?
2. How are bookmakers, markets, and outcomes nested?
3. What fields are present, and what are their types?
4. Are there nulls or inconsistencies to handle?
5. What does the flattened schema look like?

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

## 1. Load raw JSON
Pick the most recent file from `data/raw/`.

In [ ]:
raw_dir = Path('../data/raw')
latest_file = sorted(raw_dir.glob('tennis_odds_*.json'))[-1]
print(f'Loading: {latest_file}')

with open(latest_file, encoding='utf-8') as f:
    raw = json.load(f)

print(f'Total events: {len(raw)}')

## 2. Top-level structure
Inspect the keys of a single event and the overall shape.

In [ ]:
# Keys in a single event
sample_event = raw[0]
print('Top-level keys:', list(sample_event.keys()))

In [ ]:
# Quick DataFrame of top-level scalar fields (no nesting)
scalar_fields = ['id', 'sport_key', 'sport_title', 'commence_time', 'home_team', 'away_team']
df_events = pd.DataFrame([{k: e[k] for k in scalar_fields if k in e} for e in raw])
df_events.head(10)

## 3. Nested structure: bookmakers → markets → outcomes

In [ ]:
# How many bookmakers per event on average?
bm_counts = [len(e.get('bookmakers', [])) for e in raw]
print(f'Bookmakers per event — min: {min(bm_counts)}, max: {max(bm_counts)}, avg: {sum(bm_counts)/len(bm_counts):.1f}')

In [ ]:
# Inspect one bookmaker entry
sample_bookmaker = sample_event['bookmakers'][0]
print(json.dumps(sample_bookmaker, indent=2))

In [ ]:
# Inspect one market and its outcomes
sample_market = sample_bookmaker['markets'][0]
print(json.dumps(sample_market, indent=2))

## 4. Flatten the nested structure
Each row = one outcome (player price) from one bookmaker for one event.

In [ ]:
rows = []
for event in raw:
    event_id = event['id']
    commence_time = event['commence_time']
    home_team = event['home_team']
    away_team = event['away_team']

    for bookmaker in event.get('bookmakers', []):
        bookmaker_key = bookmaker['key']
        bookmaker_title = bookmaker['title']
        last_update = bookmaker.get('last_update')

        for market in bookmaker.get('markets', []):
            market_key = market['key']

            for outcome in market.get('outcomes', []):
                rows.append({
                    'event_id': event_id,
                    'commence_time': commence_time,
                    'home_team': home_team,
                    'away_team': away_team,
                    'bookmaker_key': bookmaker_key,
                    'bookmaker_title': bookmaker_title,
                    'last_update': last_update,
                    'market_key': market_key,
                    'outcome_name': outcome['name'],
                    'price': outcome['price'],
                })

df = pd.DataFrame(rows)
print(f'Shape: {df.shape}')
df.head(20)

## 5. Data quality checks

In [ ]:
print('Dtypes:')
print(df.dtypes)
print('\nNull counts:')
print(df.isnull().sum())

In [ ]:
# Unique bookmakers present
print('Bookmakers:', df['bookmaker_key'].unique())

# Price range sanity check
print('\nPrice stats:')
print(df['price'].describe())

## 6. Notes & schema decisions

> Fill this in after running the cells above. Document:
> - Final agreed column names and types
> - Any nulls or anomalies found
> - Transform decisions to carry into `src/processing/transform.py`
> - Then update `docs/MIAMI_OPEN_SCHEMA.md` accordingly.